<a href="https://colab.research.google.com/github/fernandodeeke/Hydraulics/blob/main/turbulent_2_tank_slide.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from ipywidgets import interact, FloatSlider, VBox, HBox, Label, HTML
import ipywidgets as widgets

# ============================================================
# TWO-TANK SYSTEM — INTERACTIVE NOTEBOOK
# Adjust sliders below to explore how each physical parameter
# changes the system's transient response and equilibrium.
# ============================================================

# ── Slider definitions ──────────────────────────────────────
# Each slider is labelled with its symbol, SI unit, and a
# short physical description so the notebook is self-contained.

slider_style  = {'description_width': '260px'}
slider_layout = widgets.Layout(width='600px')

def make_slider(desc, val, lo, hi, step, fmt='.4g'):
    return FloatSlider(
        value=val, min=lo, max=hi, step=step,
        description=desc,
        readout_format=fmt,
        style=slider_style, layout=slider_layout,
        continuous_update=True
    )

# ── Section headers ──────────────────────────────────────────
def header(text):
    return HTML(f"<b style='font-size:13px; color:#444'>{text}</b>")

# ── Initial Conditions ───────────────────────────────────────
w_h1_0 = make_slider('h₁(0)  Initial level Tank 1 (m)', 0.00, 0.00, 0.40, 0.005, '.3f')
w_h2_0 = make_slider('h₂(0)  Initial level Tank 2 (m)', 0.00, 0.00, 0.40, 0.005, '.3f')

# ── Tank Geometry ────────────────────────────────────────────
w_A1 = make_slider('A₁  Cross-section Tank 1 (m²)',  0.02,  0.005, 0.10,  0.005, '.4f')
w_A2 = make_slider('A₂  Cross-section Tank 2 (m²)',  0.02,  0.005, 0.10,  0.005, '.4f')

# ── Inflow ───────────────────────────────────────────────────
# 1 L/min = 1.667e-5 m³/s.  Slider in L/min, converted inside.
w_q1_lpm = make_slider('q₁  Inflow rate (L/min)',          1.00,  0.10,  5.00,  0.10,  '.2f')

# ── Fluid Properties ─────────────────────────────────────────
w_mu  = make_slider('μ   Dynamic viscosity ×10⁻³ (Pa·s)',  1.00,  0.50,  5.00,  0.10,  '.2f')
w_rho = make_slider('ρ   Fluid density (kg/m³)',         1000.0, 800.0, 1200.0, 10.0,  '.1f')

# ── Connecting Pipe (Laminar, Hagen–Poiseuille) ──────────────
w_L = make_slider('L   Pipe length (m)',                   0.20,  0.05,  1.00,  0.01,  '.3f')
w_r = make_slider('r   Pipe radius ×10⁻³ (m)',             1.50,  0.50,  5.00,  0.10,  '.2f')

# ── Outlet Valve (Turbulent, Torricelli) ─────────────────────
w_Cd = make_slider('Cₐ  Discharge coefficient (—)',        0.62,  0.30,  0.90,  0.01,  '.3f')
w_a  = make_slider('a   Valve area ×10⁻⁵ (m²)',            1.50,  0.10,  5.00,  0.10,  '.2f')

# ── Simulation horizon ───────────────────────────────────────
w_T  = make_slider('T   Simulation horizon (s)',         1500.0, 300.0, 3600.0, 60.0,  '.0f')


# ============================================================
# Core simulation function
# All parameters arrive in SI units.
# ============================================================
def run_simulation(h1_0, h2_0, A1, A2, q1, mu, rho, L, r, Cd, a, T):
    g = 9.81  # gravitational acceleration (m/s²) — fixed constant

    # ── Derived coefficients ─────────────────────────────────
    # Hagen–Poiseuille: laminar resistance of the connecting pipe
    #   Q = ΔP / (8μL / πr⁴)  →  in head form: q₁₂ = (h₁−h₂) / R₁₂
    R12 = (8 * mu * L) / (rho * g * np.pi * r**4)

    # Torricelli (turbulent): outlet flow = Cd·a·√(2g·h₂)
    #   lumped coefficient absorbs everything except √h₂
    k2 = Cd * a * np.sqrt(2 * g)

    # ── Analytical equilibrium ───────────────────────────────
    # At steady state dh/dt = 0:
    #   q₁ = k₂·√h₂*  →  h₂* = (q₁/k₂)²
    #   q₁ = (h₁*−h₂*)/R₁₂  →  h₁* = R₁₂·q₁ + h₂*
    h2_star = (q1 / k2) ** 2
    h1_star = R12 * q1 + h2_star

    # ── ODE system ──────────────────────────────────────────
    def two_tanks(t, h):
        h1, h2 = max(0.0, h[0]), max(0.0, h[1])   # physical floor

        q12 = (h1 - h2) / R12      # laminar  (linear in Δh)
        q2  = k2 * np.sqrt(h2)     # turbulent (nonlinear in h₂)

        dh1_dt = (q1 - q12) / A1   # mass balance, Tank 1
        dh2_dt = (q12 - q2)  / A2  # mass balance, Tank 2
        return [dh1_dt, dh2_dt]

    # ── Numerical integration (RK45) ────────────────────────
    t_eval = np.linspace(0, T, 1000)
    sol = solve_ivp(two_tanks, (0, T), [h1_0, h2_0],
                    t_eval=t_eval, method='RK45', max_step=T/500)

    return sol, h1_star, h2_star, R12, k2


# ============================================================
# Interactive plot function — called by ipywidgets on every
# slider change.
# ============================================================
def plot(h1_0, h2_0, A1, A2, q1_lpm, mu_mPas, rho, L, r_mm, Cd, a_1e5, T):

    # Unit conversions from slider-friendly to SI
    q1 = q1_lpm  * 1e-3 / 60        # L/min  → m³/s
    mu = mu_mPas * 1e-3              # mPa·s  → Pa·s
    r  = r_mm    * 1e-3              # mm     → m
    a  = a_1e5   * 1e-5              # ×10⁻⁵  → m²

    sol, h1_star, h2_star, R12, k2 = run_simulation(
        h1_0, h2_0, A1, A2, q1, mu, rho, L, r, Cd, a, T
    )

    # ── Plotting ─────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(10, 5))

    # Transient responses (convert m → cm for readability)
    ax.plot(sol.t, sol.y[0] * 100, 'steelblue',   lw=2.0, label='Tank 1  $h_1(t)$')
    ax.plot(sol.t, sol.y[1] * 100, 'darkorange',  lw=2.0, label='Tank 2  $h_2(t)$')

    # Equilibrium reference lines
    ax.axhline(h1_star * 100, color='steelblue',  lw=1.2, ls='--', alpha=0.6,
               label=f'Steady state  $h_1^*$ = {h1_star*100:.2f} cm')
    ax.axhline(h2_star * 100, color='darkorange', lw=1.2, ls='--', alpha=0.6,
               label=f'Steady state  $h_2^*$ = {h2_star*100:.2f} cm')

    ax.set_xlabel('Time  (s)',         fontsize=12)
    ax.set_ylabel('Water level  (cm)', fontsize=12)
    ax.set_title('Two-Tank System — Transient Response', fontsize=13)
    ax.legend(loc='lower right', fontsize=10)
    ax.grid(True, ls=':', alpha=0.5)
    ax.set_xlim(0, T)

    # ── Derived-parameter annotation ─────────────────────────
    info = (f'R₁₂ = {R12:.2f} s/m²   '
            f'k₂ = {k2:.2e} m²·⁵/s   '
            f'h₁* = {h1_star*100:.2f} cm   '
            f'h₂* = {h2_star*100:.2f} cm')
    ax.text(0.02, 0.97, info, transform=ax.transAxes, fontsize=9,
            va='top', color='#333',
            bbox=dict(boxstyle='round,pad=0.3', fc='white', alpha=0.7))

    plt.tight_layout()
    plt.show()


# ============================================================
# Widget layout — group sliders by physical meaning
# ============================================================
ui = VBox([
    header('── Initial Conditions ───────────────────────────────'),
    w_h1_0, w_h2_0,

    header('── Tank Geometry ────────────────────────────────────'),
    w_A1, w_A2,

    header('── Inflow ───────────────────────────────────────────'),
    w_q1_lpm,

    header('── Fluid Properties ─────────────────────────────────'),
    w_mu, w_rho,

    header('── Connecting Pipe  (Hagen–Poiseuille) ─────────────'),
    w_L, w_r,

    header('── Outlet Valve  (Torricelli / turbulent) ───────────'),
    w_Cd, w_a,

    header('── Simulation ───────────────────────────────────────'),
    w_T,
])

out = widgets.interactive_output(plot, {
    'h1_0':    w_h1_0,
    'h2_0':    w_h2_0,
    'A1':      w_A1,
    'A2':      w_A2,
    'q1_lpm':  w_q1_lpm,
    'mu_mPas': w_mu,
    'rho':     w_rho,
    'L':       w_L,
    'r_mm':    w_r,
    'Cd':      w_Cd,
    'a_1e5':   w_a,
    'T':       w_T,
})

# ── Display everything ───────────────────────────────────────
display(ui, out)

Output()